In [23]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, r2_score , mean_absolute_error
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso , ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from xgboost import XGBRegressor 
from catboost import CatBoostRegressor
from sklearn.svm import SVR





In [24]:
df=pd.read_csv('..\\notebook\\data\\StudentsPerformance.csv')

In [25]:
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [26]:
x=df.drop(columns=['math score'])
y=df['math score']

In [27]:
num_features=x.select_dtypes(include=['int64', 'float64']).columns
cat_features=x.select_dtypes(include=['str']).columns

In [28]:
print(type(num_features))
print(type(cat_features))

<class 'pandas.Index'>
<class 'pandas.Index'>


In [29]:
num_features

Index(['reading score', 'writing score'], dtype='str')

In [30]:
ss = StandardScaler()
ohe = OneHotEncoder()
ct=ColumnTransformer(transformers=[
    ('onehotencoding', ohe, cat_features),
    ('standarscaler',ss, num_features),
])



In [31]:
x=ct.fit_transform(x)


In [32]:
# separate the dataset into train and test split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)  


##### Evaluation funcion to give all metrics after model training

In [33]:
def evaluate_model(true,predicted):
    mse=mean_squared_error(true,predicted)
    r2=r2_score(true,predicted)
    mae=mean_absolute_error(true,predicted)
    return mse,r2,mae
    

In [34]:
models={
    "LinearRegression" : LinearRegression(),
    "Ridge" : Ridge(),
    "Lasso" : Lasso(),
    "RandomForestRegressor" : RandomForestRegressor(),
    "CatBoostRegressor" : CatBoostRegressor(verbose=0),
    "xgboost" : XGBRegressor(),
    "SVR" : SVR(),
    "ElasticNet" : ElasticNet(),
    "DecisionTreeRegressor" : DecisionTreeRegressor(),
    "KNeighborsRegressor" : KNeighborsRegressor(),
    "GradientBoostingRegressor" : GradientBoostingRegressor(),
}

model_list=[]
r2_list=[]

for i in range(len(models)):
    model=list(models.values())[i]
    model.fit(x_train,y_train)
    y_train_pred=model.predict(x_train)
    y_test_pred=model.predict(x_test)

    # Evaluate the model on train and test data
    train_mse, train_r2, train_mae = evaluate_model(y_train, y_train_pred)
    test_mse, test_r2, test_mae = evaluate_model(y_test, y_test_pred)

    print(list(models.keys())[i])
    print("Train MSE:", train_mse)
    print("Train R2 Score:", train_r2)
    print("Train MAE:", train_mae)
    print("Test MSE:", test_mse)
    print("Test R2 Score:", test_r2)
    print("Test MAE:", test_mae)
    print("\n")

    model_list.append(list(models.keys())[i])
    r2_list.append(test_r2)

LinearRegression
Train MSE: 28.33487038064859
Train R2 Score: 0.8743172040139593
Train MAE: 4.266711846071957
Test MSE: 29.095169866715487
Test R2 Score: 0.8804332983749565
Test MAE: 4.21476314247485


Ridge
Train MSE: 28.33778823308244
Train R2 Score: 0.8743042615212909
Train MAE: 4.264987823725981
Test MSE: 29.056272192348324
Test R2 Score: 0.8805931485028737
Test MAE: 4.211100688014261


Lasso
Train MSE: 43.47840400585581
Train R2 Score: 0.8071462015863455
Train MAE: 5.206302661246528
Test MSE: 42.50641683841164
Test R2 Score: 0.8253197323627852
Test MAE: 5.157881810347764


RandomForestRegressor
Train MSE: 5.390223592881945
Train R2 Score: 0.976091001545361
Train MAE: 1.8331885416666664
Test MSE: 35.51453498958333
Test R2 Score: 0.8540528951058143
Test MAE: 4.6410875


CatBoostRegressor
Train MSE: 9.257805405523678
Train R2 Score: 0.9589358676277713
Train MAE: 2.405393926779502
Test MSE: 36.10365799356841
Test R2 Score: 0.8516318920747058
Test MAE: 4.612531714976557


xgboost
Train

In [35]:
pd.DataFrame(list(zip(model_list,r2_list)),columns=['Model','R2 Score']).sort_values(by='R2 Score',ascending=False )

,Model,R2 Score
1,Ridge,0.880593
0,LinearRegression,0.880433
10,GradientBoostingRegressor,0.872476
3,RandomForestRegressor,0.854053
4,CatBoostRegressor,0.851632
5,xgboost,0.827797
2,Lasso,0.825320
9,KNeighborsRegressor,0.783813
7,ElasticNet,0.739624
8,DecisionTreeRegressor,0.738903


#### Ridge Regression

In [37]:
r_model=Ridge()
r_model.fit(x_train,y_train)
predicted_output=r_model.predict(x_test)
r2score=r2_score(y_test,predicted_output)*100
print("Ridge Regression R2 Score:", r2score)

Ridge Regression R2 Score: 88.05931485028738


In [41]:
# Difference between predcted and actual values
pred_df=pd.DataFrame({'Actual':y_test,'predicted':predicted_output,'difference':np.abs(y_test-predicted_output)})
print(pred_df)

     Actual  predicted  difference
521      91  76.398705   14.601295
737      53  58.842469    5.842469
740      80  76.992129    3.007871
660      74  76.805898    2.805898
411      84  87.615387    3.615387
..      ...        ...         ...
408      52  43.492293    8.507707
332      62  62.170763    0.170763
208      74  67.910325    6.089675
613      65  67.061504    2.061504
78       61  62.427189    1.427189

[200 rows x 3 columns]
